In [2]:
import polars as pl
import pandas as pd

In [3]:
test_df = pl.read_csv('/home/dangnh36/datasets/ecg/raw/test.csv')
test_df

id,lead,fs,number_of_rows
i64,str,i64,i64
1053922973,"""I""",1000,2500
1053922973,"""II""",1000,10000
1053922973,"""III""",1000,2500
1053922973,"""aVR""",1000,2500
1053922973,"""aVL""",1000,2500
…,…,…,…
2352854581,"""V2""",1000,2500
2352854581,"""V3""",1000,2500
2352854581,"""V4""",1000,2500


In [7]:
print(test_df['lead'].to_list())

['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']


In [18]:
df = pl.read_csv('/home/dangnh36/datasets/ecg/raw/train.csv')
df

id,fs,sig_len
i64,i64,i64
7663343,500,5000
10140238,1000,10000
11842146,1000,10000
19030958,250,2500
19585145,512,5120
…,…,…
4277555223,1000,10000
4283871417,256,2560
4284351157,512,5120


In [21]:
import math


new_rows = []
for row in df.iter_rows(named = True):
    sample_id = row['id']
    for type_id in [1, 3, 4, 5, 6, 9, 10, 11, 12]:
        new_id = f'{sample_id}/{sample_id}-{type_id:04d}'
        for lead_name in ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']:
            new_rows.append({'id': new_id, 'lead': lead_name, 'fs': row['fs'], 'number_of_rows': math.floor(row['fs'] * 2.5) if lead_name != 'II' else math.floor(row['fs'] * 10), 'sample_id': sample_id})

new_df = pl.DataFrame(new_rows)
new_df

id,lead,fs,number_of_rows,sample_id
str,str,i64,i64,i64
"""7663343/7663343-0001""","""I""",500,1250,7663343
"""7663343/7663343-0001""","""II""",500,5000,7663343
"""7663343/7663343-0001""","""III""",500,1250,7663343
"""7663343/7663343-0001""","""aVR""",500,1250,7663343
"""7663343/7663343-0001""","""aVL""",500,1250,7663343
…,…,…,…,…
"""4292118763/4292118763-0012""","""V2""",512,1280,4292118763
"""4292118763/4292118763-0012""","""V3""",512,1280,4292118763
"""4292118763/4292118763-0012""","""V4""",512,1280,4292118763


In [22]:
new_df.write_csv('/home/dangnh36/datasets/ecg/processed/pseudo_test.csv')

In [23]:
import json

with open('/home/dangnh36/datasets/ecg/processed/cv/v1/skf_1x5_rd42.json', 'r') as f:
    cv_meta = json.load(f)

fold_val_ids = cv_meta['folds'][0]['val_ids']
print(fold_val_ids)

[11842146, 31294838, 57655084, 102150619, 112870634, 129883643, 136433453, 136524867, 144746082, 145399852, 157249266, 162622667, 198290386, 202613567, 215513144, 225208096, 230071334, 235857261, 240241572, 242807775, 264050025, 309723963, 322694441, 327451863, 331919502, 334628058, 347249497, 370507243, 505886807, 514629128, 540379822, 541485763, 548033375, 601889565, 604355197, 610253198, 622992199, 640106434, 644026427, 734480510, 735091708, 752641433, 783737093, 828004421, 830220007, 831376508, 831662714, 835070631, 847329431, 866477990, 887374646, 906806490, 1006427285, 1026034238, 1041099777, 1079294623, 1094825522, 1103158012, 1135737846, 1151562032, 1154001412, 1195437044, 1197069178, 1199125619, 1209416227, 1219936451, 1223595426, 1281093472, 1306725887, 1376451566, 1381006600, 1445349505, 1446634391, 1484831977, 1509005816, 1533451101, 1539510874, 1542217251, 1556112478, 1589810571, 1643754023, 1649759478, 1671648942, 1692881983, 1717124873, 1734729516, 1739475663, 1774656418

In [27]:
fold_new_df = new_df.filter(pl.col('sample_id').is_in(fold_val_ids))
fold_new_df.write_csv('/home/dangnh36/datasets/ecg/processed/pseudo_test_fold0.csv')
fold_new_df

id,lead,fs,number_of_rows,sample_id
str,str,i64,i64,i64
"""11842146/11842146-0001""","""I""",1000,2500,11842146
"""11842146/11842146-0001""","""II""",1000,10000,11842146
"""11842146/11842146-0001""","""III""",1000,2500,11842146
"""11842146/11842146-0001""","""aVR""",1000,2500,11842146
"""11842146/11842146-0001""","""aVL""",1000,2500,11842146
…,…,…,…,…
"""4284351157/4284351157-0012""","""V2""",512,1280,4284351157
"""4284351157/4284351157-0012""","""V3""",512,1280,4284351157
"""4284351157/4284351157-0012""","""V4""",512,1280,4284351157


In [29]:
fold_new_df = new_df.filter(pl.col('sample_id').is_in(fold_val_ids))[:1200 * 12]
fold_new_df.write_csv('/home/dangnh36/datasets/ecg/processed/pseudo_test_fold0_1200images.csv')
fold_new_df

id,lead,fs,number_of_rows,sample_id
str,str,i64,i64,i64
"""11842146/11842146-0001""","""I""",1000,2500,11842146
"""11842146/11842146-0001""","""II""",1000,10000,11842146
"""11842146/11842146-0001""","""III""",1000,2500,11842146
"""11842146/11842146-0001""","""aVR""",1000,2500,11842146
"""11842146/11842146-0001""","""aVL""",1000,2500,11842146
…,…,…,…,…
"""2818726012/2818726012-0004""","""V2""",1025,2562,2818726012
"""2818726012/2818726012-0004""","""V3""",1025,2562,2818726012
"""2818726012/2818726012-0004""","""V4""",1025,2562,2818726012


In [30]:
[e - 2365 for e in [2391, 2369, 2386, 2406]]

[26, 4, 21, 41]